In [40]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0], 
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 
emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"




In [ ]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [42]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


In [43]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)



-43.45111319916465


In [44]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [45]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


-42.58225552052512


In [46]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [47]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [48]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides. 

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .] 
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .] 
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [49]:
def setup_viterbi(query_sequence):

    num_states = len(states)
    seq_len = len(query_sequence)

    viterbi_value_matrix = np.full((num_states, seq_len),-float("inf"))

    viterbi_trace_matrix = np.full((num_states, seq_len),-1,dtype=int)

    first_nuc = query_sequence[0]

    for state_name, state_id in states.items():

        emission_prob = emission_probs[state_id][emission_nuc_codes[first_nuc]]

        if emission_prob > 0:

            viterbi_value_matrix[state_id][0] = (
                math.log(0.25) + math.log(emission_prob)
            )

        # initialize trace matrix
        viterbi_trace_matrix[state_id][0] = state_id

    return viterbi_value_matrix, viterbi_trace_matrix



### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND** 

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [53]:
def viterbi_forward(query_sequence,viterbi_value_matrix,viterbi_trace_matrix):

    seq_len = len(query_sequence)

    for col in range(1, seq_len):

        current_nuc = query_sequence[col]

        for curr_state_name, curr_state_id in states.items():

            emission_prob = emission_probs[curr_state_id][emission_nuc_codes[current_nuc]]

            if emission_prob == 0:
                continue

            max_prob = -float("inf")
            best_prev_state = -1

            for prev_state_name, prev_state_id in states.items():

                transition_prob = state_transition_prob[prev_state_id][curr_state_id]

                if transition_prob == 0:
                    continue

                prev_prob = viterbi_value_matrix[prev_state_id][col - 1]

                if prev_prob == -float("inf"):
                    continue

                curr_prob = (prev_prob+ math.log(transition_prob)+ math.log(emission_prob))

                if curr_prob > max_prob:
                    max_prob = curr_prob
                    best_prev_state = prev_state_id

            viterbi_value_matrix[curr_state_id][col] = max_prob
            viterbi_trace_matrix[curr_state_id][col] = best_prev_state


In [54]:
def traceback_best_state_path(viterbi_value_matrix,viterbi_trace_matrix):

    rows, cols = viterbi_value_matrix.shape

    last_state = np.argmax(viterbi_value_matrix[:, cols - 1])

    best_path = [last_state]

    current_state = last_state

    for col in range(cols - 1, 0, -1):

        current_state = viterbi_trace_matrix[current_state][col]

        best_path.append(current_state)

    best_path.reverse()

    best_state_sequence = [
        id2state[state_id]
        for state_id in best_path
    ]

    return best_state_sequence


In [55]:
viterbi_value_matrix, viterbi_trace_matrix = setup_viterbi(query_sequence)
viterbi_forward(query_sequence,viterbi_value_matrix,viterbi_trace_matrix)
best_state_path = traceback_best_state_path(viterbi_value_matrix,viterbi_trace_matrix)
print(best_state_path)

['E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E']
